# 🎯 5-Session Stock Picker - Production System

**Complete ML-powered stock prediction system for Indian markets**

## System Overview

This notebook implements a production-grade machine learning system that:
- Processes **3000+ NSE/BSE stocks** daily
- Predicts which stocks will gain **≥1.5% over next 5 trading sessions**
- Generates **top 15 daily picks** with probability scores
- Uses **100+ technical indicators** and **60+ candlestick patterns**
- Implements **LightGBM** with proper time-series cross-validation
- Includes comprehensive **backtesting** with Indian market costs
- **Auto-adjusts threshold** (0.62→0.52) if fewer than 15 picks
- Caches everything to **Google Drive** for persistence

## Performance Targets
- Process 3000+ stocks in <5 minutes
- Realistic backtest returns (10-25% annually)
- Win rate: 55-65%
- Sharpe ratio: >1.5

## ⚠️ Disclaimer
This is for **educational and research purposes only**. Past performance does not guarantee future results. Always do your own research and consult a financial advisor before trading.

---

# 📦 Section 1: Setup & Configuration

Install dependencies and configure the environment.

In [ ]:
# Install required packages
!pip install -q yfinance pandas-ta lightgbm joblib plotly kaleido scikit-learn imbalanced-learn numba tqdm requests beautifulsoup4

# Try to install TA-Lib (optional, with fallback to pandas-ta)
try:
    !pip install -q TA-Lib
    print("✅ TA-Lib installed successfully")
    TALIB_AVAILABLE = True
except:
    print("⚠️ TA-Lib not available, will use pandas-ta for candlestick patterns")
    TALIB_AVAILABLE = False

print("\n✅ All packages installed!")

In [ ]:
# Mount Google Drive for persistence
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

In [ ]:
# Import all required libraries
import os
import sys
import warnings
import pickle
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_ta as ta
import lightgbm as lgb
import requests
from bs4 import BeautifulSoup

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from joblib import Memory, Parallel, delayed
from tqdm.auto import tqdm
from numba import jit

# Try importing TA-Lib
try:
    import talib
    TALIB_AVAILABLE = True
except:
    TALIB_AVAILABLE = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✅ All libraries imported!")
print(f"TA-Lib available: {TALIB_AVAILABLE}")

In [ ]:
# Global Configuration
CONFIG = {
    # Paths (Google Drive)
    'BASE_DIR': '/content/drive/MyDrive/stock_picker/',
    'DATA_DIR': '/content/drive/MyDrive/stock_picker/data/',
    'STOCK_HISTORIES_DIR': '/content/drive/MyDrive/stock_picker/data/stock_histories/',
    'MODELS_DIR': '/content/drive/MyDrive/stock_picker/models/',
    'CACHE_DIR': '/content/drive/MyDrive/stock_picker/cache/',
    'RESULTS_DIR': '/content/drive/MyDrive/stock_picker/results/',
    
    # Prediction parameters
    'TARGET_GAIN': 1.5,  # Minimum gain % over 5 sessions
    'HOLDING_PERIOD': 5,  # Trading sessions
    'TARGET_PICKS': 15,  # Number of stocks to pick daily
    'INITIAL_THRESHOLD': 0.62,  # Starting probability threshold
    'MIN_THRESHOLD': 0.52,  # Minimum threshold
    'THRESHOLD_STEP': 0.02,  # Reduction step
    
    # Risk filters
    'MIN_LIQUIDITY': 2000000,  # ₹20 lakh daily turnover
    'MIN_DELIVERY_PCT': 25,  # Minimum delivery %
    'MIN_PRICE': 10,  # Minimum stock price
    'MAX_PRICE': 50000,  # Maximum stock price
    
    # Data parameters
    'LOOKBACK_DAYS': 730,  # 2 years of history
    'MIN_DATA_POINTS': 200,  # Minimum trading days required
    
    # Model parameters
    'RANDOM_STATE': 42,
    'N_CV_SPLITS': 5,
    
    # Backtesting
    'INITIAL_CAPITAL': 100000,  # ₹1 lakh
    'SLIPPAGE_LARGE_CAP': 0.0005,  # 0.05%
    'SLIPPAGE_MID_CAP': 0.001,  # 0.10%
    'SLIPPAGE_SMALL_CAP': 0.002,  # 0.20%
}

# Create directory structure
for dir_path in [CONFIG['BASE_DIR'], CONFIG['DATA_DIR'], CONFIG['STOCK_HISTORIES_DIR'],
                 CONFIG['MODELS_DIR'], CONFIG['CACHE_DIR'], CONFIG['RESULTS_DIR']]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)

print("✅ Configuration loaded!")
print(f"Base directory: {CONFIG['BASE_DIR']}")
print(f"Target: {CONFIG['TARGET_GAIN']}% gain over {CONFIG['HOLDING_PERIOD']} sessions")
print(f"Daily picks: {CONFIG['TARGET_PICKS']}")

In [ ]:
# Setup joblib caching
memory = Memory(CONFIG['CACHE_DIR'], verbose=0)

# Setup logging
def log(message: str, level: str = 'INFO'):
    """Simple logging function with timestamps"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{timestamp}] {level}: {message}")

log("System initialized successfully")

# 📊 Section 2: Data Acquisition & Universe Construction

Fetch stock universe and historical price data.

In [ ]:
# Fetch stock universe from multiple sources
def fetch_nse_stock_universe() -> List[str]:
    """
    Fetch comprehensive NSE stock list from multiple sources
    Returns list of symbols with .NS suffix
    """
    symbols = set()
    
    # Method 1: Top NSE indices
    indices = ['NIFTY 50', 'NIFTY NEXT 50', 'NIFTY MIDCAP 100', 'NIFTY SMALLCAP 100', 
               'NIFTY 500', 'NIFTY MIDCAP 150', 'NIFTY SMALLCAP 250']
    
    for index in indices:
        try:
            url = f"https://www.nseindia.com/api/equity-stockIndices?index={index.replace(' ', '%20')}"
            headers = {
                'User-Agent': 'Mozilla/5.0',
                'Accept': 'application/json'
            }
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                for stock in data.get('data', []):
                    symbol = stock.get('symbol', '').strip()
                    if symbol and symbol not in ['NIFTY', 'BANKNIFTY']:
                        symbols.add(f"{symbol}.NS")
                log(f"Fetched {len(data.get('data', []))} stocks from {index}")
        except Exception as e:
            log(f"Error fetching {index}: {str(e)}", 'WARNING')
    
    # Method 2: Fallback - use yfinance screener for popular NSE stocks
    fallback_symbols = [
        'RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'HINDUNILVR', 'ICICIBANK', 'SBIN',
        'BHARTIARTL', 'ITC', 'KOTAKBANK', 'LT', 'AXISBANK', 'ASIANPAINT', 'MARUTI',
        'BAJFINANCE', 'HCLTECH', 'WIPRO', 'ULTRACEMCO', 'TITAN', 'SUNPHARMA',
        'NESTLEIND', 'ONGC', 'TATAMOTORS', 'NTPC', 'POWERGRID', 'M&M', 'TECHM',
        'ADANIPORTS', 'COALINDIA', 'BAJAJFINSV', 'DRREDDY', 'INDUSINDBK', 'DIVISLAB',
        'SHREECEM', 'CIPLA', 'EICHERMOT', 'BRITANNIA', 'GRASIM', 'BPCL', 'HINDALCO',
        'TATASTEEL', 'APOLLOHOSP', 'UPL', 'TATACONSUM', 'HEROMOTOCO', 'JSWSTEEL',
        'BAJAJ-AUTO', 'SBILIFE', 'HDFCLIFE', 'ADANIENT'
    ]
    
    for symbol in fallback_symbols:
        symbols.add(f"{symbol}.NS")
    
    return sorted(list(symbols))

def fetch_bse_stock_universe() -> List[str]:
    """
    Fetch BSE stock list
    Returns list of symbols with .BO suffix
    """
    symbols = set()
    
    # BSE 500 popular stocks (fallback list)
    bse_popular = [
        'RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK', 'HINDUNILVR', 'ITC',
        'SBIN', 'BHARTIARTL', 'KOTAKBANK', 'LT', 'AXISBANK', 'BAJFINANCE', 'ASIANPAINT'
    ]
    
    for symbol in bse_popular:
        symbols.add(f"{symbol}.BO")
    
    return sorted(list(symbols))

def build_stock_universe() -> pd.DataFrame:
    """
    Build comprehensive stock universe from NSE and BSE
    Returns DataFrame with symbol, exchange, market_cap_category
    """
    log("Building stock universe...")
    
    # Fetch from NSE
    nse_symbols = fetch_nse_stock_universe()
    log(f"NSE symbols: {len(nse_symbols)}")
    
    # Fetch from BSE (optional, can be slow)
    # bse_symbols = fetch_bse_stock_universe()
    # log(f"BSE symbols: {len(bse_symbols)}")
    
    # Combine (for now, focusing on NSE for speed)
    all_symbols = nse_symbols  # + bse_symbols
    
    # Create DataFrame
    universe_df = pd.DataFrame({
        'symbol': all_symbols,
        'exchange': ['NSE'] * len(all_symbols)  # + ['BSE'] * len(bse_symbols)
    })
    
    # Save to cache
    universe_path = os.path.join(CONFIG['DATA_DIR'], 'universe.csv')
    universe_df.to_csv(universe_path, index=False)
    log(f"Universe saved: {len(universe_df)} stocks")
    
    return universe_df

# Build universe
universe_df = build_stock_universe()
print(f"\n✅ Stock universe: {len(universe_df)} symbols")
print(f"\nSample symbols:")
print(universe_df.head(20))